# Continuum subtraction in the uv domain (uv_continuum_subtraction)

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/visibility_manipulation/uv_continuum_subtraction.ipynb)

`uv_continuum_subtraction` is the port of the ALMA and VLA pipelines' `hif_uvcontsub`: every visibility spectrum (one time, baseline and polarization) has a polynomial in frequency fitted to its continuum channels and subtracted from all its channels, as CASA `uvcontsub` does. The continuum-subtracted visibilities go into a new data group of the same processing set, ready for cube imaging of the lines.

The pipeline takes the continuum ranges from `cont.dat`, written by `hif_findcont`. Here they come from `find_continuum_ultra`: `continuum_fit_from_find_continuum` turns its result for a spectral window into the continuum frequency ranges, their frame and the fit order, with the pipeline's rule (order 1, or 0 when the continuum finding flagged a low bandwidth or a low spread of the continuum channels).

This notebook simulates a small ALMA data set with spectral lines at known channels in two spectral windows, over a continuum that changes across the band; images it, finds the continuum channels of each window with `find_continuum_ultra`, subtracts the continuum in the uv domain, images the line data and compares them with the simulation and with CASA `uvcontsub`. It runs in about a minute.

---
## Pseudo Code

```
distributed_applications/visibility_manipulation/uv_continuum_subtraction(ps_store, spectral_window_name,
                                    continuum_frequency_ranges, continuum_frequency_frame, fit_order, ...)
    open the processing set lazily; check the MSv4s of the spectral window
    Doppler factors of their times when the ranges are in another frame (TOPO MSv4, LSRK ranges)
    create the output variables (correlated data, flags) in every MSv4
    map over blocks of times of every MSv4                          (node task)
        load the input data group of the block
        continuum channels at every time: the channel frequency, converted to the frame of the ranges
        fit a weighted polynomial to the valid continuum samples of every spectrum
        subtract it from all channels; flag the spectra without a valid continuum sample
        write the block
    register the output data group; return the fit statistics, mean spectra, timing
```

## Notes

- The fit follows CASA `uvcontsub` (`fitmethod="gsl"`): weighted least squares with the data weights, the real and imaginary parts with the same weights, the order lowered to `n - 1` for a spectrum with `n <= fit_order` valid channels, the polynomial subtracted from every channel; flags of fitted spectra and weights are unchanged.
- Departures from CASA and the pipeline:
    - the continuum channels are found at every time, so a channel is used only at the times its frequency lies in a range; the pipeline fits one TOPO channel selection per measurement set at all times, the channels that overlap a range at any time;
    - a spectrum without a valid continuum channel is flagged instead of being kept unsubtracted;
    - samples of zero weight or with a non-finite visibility or weight are not valid channels;
    - the fit takes the weights of the data group (XRADIO's `WEIGHT`); CASA weights a fit of the `DATA` column by `1 / SIGMA_SPECTRUM**2`.
- One call per spectral window, for all its MSv4s; the output is a data group of the same processing set, not a new measurement set.

## Install AstroVIPER

Skip this cell when AstroVIPER is already installed.

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401 -- availability probe
except ImportError:
    os.system("pip install --upgrade astroviper[all]")
    import astroviper  # noqa: F401 -- verify the installation

print("Using astroviper version", version("astroviper"))

---
## API

In [ ]:
from astroviper.distributed_applications.imaging import (
    find_continuum_ultra,
    image_cube_single_field,
)
from astroviper.distributed_applications.visibility_manipulation import (
    uv_continuum_subtraction,
)
from astroviper.processing_functions.visibility_manipulation import (
    continuum_fit_from_find_continuum,
)

uv_continuum_subtraction?

## Example: two simulated ALMA spectral windows

### A Dask cluster

Four worker processes with one thread each. The log level is WARNING to keep the output short; at INFO the applications log their timing.

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(
    cores=4, memory_limit="4GB", log_params={"log_level": "WARNING"}
)

### The simulated data set

ALMA antennas observe the field around its transit. A continuum point source sits at the phase centre and a second, brighter one 3.6 arcseconds away with a spectral index of 2.5, so that the continuum visibilities change in amplitude and phase across the band. Four spectral lines (Gaussian in frequency, Gaussian sources on the sky) lie at known channels of two spectral windows. The simulator adds thermal noise.

In [ ]:
import numpy as np
import pandas as pd

ARCSEC = np.pi / (180.0 * 3600.0)
PHASE_CENTER = np.array([1.2, -0.4])  # radians
N_ANTENNA, N_CHANNELS = 20, 160
CHANNEL_WIDTH = 0.9765625e6  # Hz
CONTINUUM = [
    # flux at the first channel of spw_0 (Jy), spectral index, offset east and north (arcsec)
    {"flux_jy": 0.015, "spectral_index": 0.0, "offset_arcsec": (0.0, 0.0)},
    {"flux_jy": 0.030, "spectral_index": 2.5, "offset_arcsec": (3.0, -2.0)},
]
WINDOWS = [
    {"name": "spw_0", "first_frequency": 230.0e9},
    {"name": "spw_1", "first_frequency": 231.5e9},
]
LINES = pd.DataFrame(
    [
        (0, 0.08, 40, 8, 0.0, 0.0, 1.0, 0.8),
        (0, 0.15, 112, 20, 4.0, -3.0, 4.0, 3.0),
        (1, 0.06, 70, 12, 0.0, 0.0, 1.0, 0.8),
        (1, 0.05, 128, 4, -2.0, 2.5, 0.6, 0.6),
    ],
    columns=[
        "window",
        "peak_flux_jy",
        "channel",
        "fwhm_channels",
        "offset_east_arcsec",
        "offset_north_arcsec",
        "major_arcsec",
        "minor_arcsec",
    ],
)


def line_spectrum(line):
    # Flux (Jy) of a line at every channel of its window.
    sigma = line.fwhm_channels / np.sqrt(8.0 * np.log(2.0))
    channels = np.arange(N_CHANNELS)
    return line.peak_flux_jy * np.exp(-0.5 * ((channels - line.channel) / sigma) ** 2)


LINES

In [ ]:
import shutil

from xradio.measurement_set import open_processing_set

from astroviper.distributed_applications.simulation import simulate_processing_set
from astroviper.utils.beam_models import airy_disk_model
from astroviper.utils.telescope_layout import read_telescope_layout


def correlations(spectrum):
    # XX, XY, YX and YY flux of an unpolarized spectrum.
    zeros = np.zeros_like(spectrum)
    return np.stack([spectrum, zeros, zeros, spectrum], axis=-1)


def sky_position(offset_east_arcsec, offset_north_arcsec):
    offset = np.array(
        [offset_east_arcsec / np.cos(PHASE_CENTER[1]), offset_north_arcsec]
    )
    return PHASE_CENTER + offset * ARCSEC


def simulate_window(index, ps_store):
    # The continuum sources and the lines of one spectral window.
    frequency = WINDOWS[index]["first_frequency"] + CHANNEL_WIDTH * np.arange(
        N_CHANNELS
    )
    components = [
        {
            "kind": "point",
            "flux": correlations(
                source["flux_jy"]
                * (frequency / WINDOWS[0]["first_frequency"])
                ** source["spectral_index"]
            ),
            "ra_dec": sky_position(*source["offset_arcsec"]),
        }
        for source in CONTINUUM
    ]
    for line in LINES[LINES.window == index].itertuples():
        components.append(
            {
                "kind": "gaussian",
                "flux": correlations(line_spectrum(line)),
                "ra_dec": sky_position(
                    line.offset_east_arcsec, line.offset_north_arcsec
                ),
                "major": line.major_arcsec * ARCSEC,
                "minor": line.minor_arcsec * ARCSEC,
                "pa": 0.5,
            }
        )
    return simulate_processing_set(
        ps_store=ps_store,
        antenna_xds=read_telescope_layout(
            "alma.cycle8.1", antenna_selection=list(range(N_ANTENNA))
        ),
        time_params={
            "time_start": "2026-03-01T22:12:00.000",
            "time_delta": 120.0,
            "n_samples": 15,
        },
        frequency_params={
            "freq_start": WINDOWS[index]["first_frequency"],
            "freq_delta": CHANNEL_WIDTH,
            "n_channels": N_CHANNELS,
            "channel_width": CHANNEL_WIDTH,
            "spectral_window_name": WINDOWS[index]["name"],
        },
        polarization=["XX", "YY"],
        sky_components=components,
        phase_center_ra_dec=PHASE_CENTER[None, :],
        beam_models=[airy_disk_model("alma")],
        beam_model_map=np.zeros(N_ANTENNA, int),
        noise_params={"random_seed": 11 + index},
        n_time_chunks=1,
        n_frequency_chunks=2,
        overwrite=True,
    )


ps_store = "uv_continuum_subtraction_demo.ps.zarr"
shutil.rmtree(ps_store, ignore_errors=True)
simulate_window(0, ps_store)
simulate_window(1, ps_store)
ps_xdt = open_processing_set(ps_store)
ps_xdt.xr_ps.summary()[
    ["name", "spw_name", "shape", "start_frequency", "end_frequency"]
]

### The dirty cube

`max_iter=0` makes the dirty cube of both windows, with the imager's first round of `find_continuum_ultra` written next to it (`find_continuum_first_round=True`). It keeps the variables `find_continuum_ultra` reads: the residual, the primary beam and the beam fits of the point spread function.

In [ ]:
IMAGE_PARAMS = {
    "image_size": [128, 128],
    "cell_size": np.array([-0.35, 0.35]) * ARCSEC,
    "phase_direction": PHASE_CENTER,
    "polarization_coords": ["I", "Q"],
    "time_coords": [0],
    "fft_padding": 1.2,
    "cpp_gridder": True,
}
CYCLE_CONTROL_PARAMS = {
    "max_iter": 0,
    "max_cycles": 0,
    "threshold": 0.0,
    "gain": 0.1,
    "psf_sidelobe_factor": 1.5,
    "max_iter_per_cycle": -1,
    "min_psf_fraction": 0.05,
    "max_psf_fraction": 0.8,
    "primary_beam_limit": 0.2,
}
frequencies = ps_xdt.xr_ps.get_freq_axis().values


def dirty_cube(image_store, data_group, **kwargs):
    return image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params={**IMAGE_PARAMS, "frequency_coords": frequencies},
        imaging_weights_params={
            "weighting": "briggs",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        cycle_control_params=CYCLE_CONTROL_PARAMS,
        image_data_variables_keep=[
            "sky_residual",
            "primary_beam",
            "beam_fit_params_point_spread_function",
        ],
        processing_set_data_group_name=data_group,
        n_mapping_parallelism={"frequency": 4},
        image_chunking={"frequency": 16},
        overwrite=True,
        **kwargs,
    )


image_store = "uv_continuum_subtraction_demo.img.zarr"
imaging = dirty_cube(image_store, "base", find_continuum_first_round=True)

### Continuum channels of each window

The frequency axis of the cube is linear within each window, so `linear_channel_ranges` gives the windows' channels. The windows have no atmospheric line; a flat atmosphere serves for both.

In [ ]:
from astroviper.processing_functions.imaging.utils.spectral_windows import (
    linear_channel_ranges,
)

window_ranges = linear_channel_ranges(frequencies)
WORK = "uv_continuum_subtraction_demo_work"
shutil.rmtree(WORK, ignore_errors=True)
continuum_finding = [
    find_continuum_ultra(
        input_image_store=image_store,
        work_directory=WORK,
        channel_range=window_range,
        first_round_store=imaging["find_continuum_first_round"]["store"],
        atmospheric_transmission=np.full(window_range[1] - window_range[0], 0.92),
        atmospheric_sky_temperature=np.full(window_range[1] - window_range[0], 27.0),
    )
    for window_range in window_ranges
]


def table(columns):
    # A table of the columns {name: {row: value}}, every cell as given.
    return pd.DataFrame(
        {name: pd.Series(rows, dtype=object) for name, rows in columns.items()}
    )


table(
    {
        window["name"]: {
            "continuum channel ranges": result["continuum_channel_ranges"],
            "continuum frequency ranges (GHz)": [
                [round(frequency / 1e9, 6) for frequency in pair]
                for pair in result["continuum_frequency_ranges"]
            ],
            "frequency frame": result["frequency_frame"],
            "all continuum": result["all_continuum"],
            "low bandwidth": result["low_bandwidth"],
            "low spread": result["low_spread"],
        }
        for window, result in zip(WINDOWS, continuum_finding, strict=True)
    }
)

### From the continuum ranges to the fit

`continuum_fit_from_find_continuum` takes the continuum frequency ranges (the outer edges of the continuum channels), names the cube's frame as XRADIO names the frame of a measurement set (`"LSRK"` is `"lsrk"`) and sets the fit order as the pipeline does.

In [ ]:
fits = {
    window["name"]: continuum_fit_from_find_continuum(result)
    for window, result in zip(WINDOWS, continuum_finding, strict=True)
}
fits

### Continuum subtraction

One call per spectral window. The simulated measurement sets are in LSRK, the frame of the ranges, so the continuum channels are the same at every time; a TOPO measurement set has its channel frequencies converted to LSRK at every time. The tables give the continuum channels of each MSv4 and the fits per polarization: the spectra fitted, those fitted with a lower order or without a valid continuum channel, and the chi-squared of the fits of the real and imaginary parts (the weighted sum of the squared residuals CASA computes for each fit, averaged over all fitted spectra here). With the simulator's weights (the inverse noise variance of each part) the average chi-squared is close to the number of continuum channels less the fitted terms.

In [ ]:
from IPython.display import display

subtraction = {
    name: uv_continuum_subtraction(
        ps_store=ps_store,
        spectral_window_name=name,
        n_mapping_parallelism={"time": 3},
        overwrite=True,
        **fit,
    )
    for name, fit in fits.items()
}
display(
    pd.concat(
        [result["continuum_channels"] for result in subtraction.values()]
    ).set_index("ms_name")
)
display(
    pd.concat([result["fit_statistics"] for result in subtraction.values()])
    .set_index(["ms_name", "polarization"])
    .round(2)
)

### The mean spectra before and after

The amplitude of the weighted mean over the times and baselines of the valid visibilities (unflagged, finite, positive weight), before and after the subtraction, with the continuum channels shaded. The mean is a vector mean, dominated by the continuum source at the phase centre and the compact lines.

In [ ]:
%matplotlib inline
from astroviper.utils.report_plots import plot_continuum_subtraction

for name, result in subtraction.items():
    for ms_name, spectra in result["mean_spectra"].items():
        display(
            plot_continuum_subtraction(
                spectra.frequency.values,
                spectra["VISIBILITY_MEAN_INPUT"].values,
                spectra["VISIBILITY_MEAN_OUTPUT"].values,
                spectra["CONTINUUM_TIME_FRACTION"].values,
                [str(label) for label in spectra.polarization.values],
                title=f"{name} ({ms_name})",
            )
        )

### The line cube

The data group `continuum_subtracted` holds the continuum-subtracted visibilities with their flags and the original weights; the imager makes the dirty line cube from it. The figure shows, for each window, a continuum channel and the peak channel of a line before and after the subtraction (Stokes I, the same color scale in each pair), and the spectra at the phase centre and at the continuum source with the spectral index.

In [ ]:
from xradio.image import open_image

line_store = "uv_continuum_subtraction_demo_line.img.zarr"
dirty_cube(line_store, "continuum_subtracted")
before = open_image(image_store).SKY_RESIDUAL.isel(time=0, polarization=0)
after = open_image(line_store).SKY_RESIDUAL.isel(time=0, polarization=0)

In [ ]:
import matplotlib.pyplot as plt

extent_arcsec = 64 * 0.35
pixel = 0.35  # arcsec


def pixel_of(offset_east_arcsec, offset_north_arcsec):
    # Image pixel (l, m) of an offset from the phase centre; l grows to the west.
    return (
        int(round(64 - offset_east_arcsec / pixel)),
        int(round(64 + offset_north_arcsec / pixel)),
    )


fig, axes = plt.subplots(2, 5, figsize=(17, 6.4), layout="constrained")
for row, (window, window_range) in enumerate(zip(WINDOWS, window_ranges, strict=True)):
    start = window_range[0]
    line = LINES[LINES.window == row].iloc[0]
    continuum_channel = (
        start + continuum_finding[row]["continuum_channel_ranges"][0][0] + 2
    )
    line_channel = start + int(line.channel)
    for column, (label, channel) in enumerate(
        [("continuum channel", continuum_channel), ("line peak", line_channel)]
    ):
        limit = float(np.abs(before.isel(frequency=channel)).max())
        for offset, (name, cube) in enumerate([("before", before), ("after", after)]):
            ax = axes[row, 2 * column + offset]
            ax.imshow(
                cube.isel(frequency=channel).values.T,
                origin="lower",
                cmap="RdBu_r",
                vmin=-limit,
                vmax=limit,
                extent=[extent_arcsec, -extent_arcsec, -extent_arcsec, extent_arcsec],
            )
            ax.set_title(
                f"{window['name']} {label} {channel - start}, {name}", fontsize=9
            )
            ax.set_xlabel("east (arcsec)")
            ax.set_ylabel("north (arcsec)")
    ax = axes[row, 4]
    channels = np.arange(window_range[1] - window_range[0])
    for name, position, color in (
        ("phase centre", pixel_of(0.0, 0.0), "#0072B2"),
        (
            "source with spectral index",
            pixel_of(*CONTINUUM[1]["offset_arcsec"]),
            "#E69F00",
        ),
    ):
        for cube, style, which in ((before, "-", "before"), (after, "--", "after")):
            spectrum = cube.isel(
                frequency=slice(*window_range), l=position[0], m=position[1]
            )
            ax.plot(
                channels,
                1e3 * spectrum.values,
                style,
                color=color,
                label=f"{name}, {which}",
            )
    ax.axhline(0.0, color="#8c959f", linewidth=0.8)
    ax.set_xlabel("channel of the window")
    ax.set_ylabel("Jy/beam x 1e3")
    ax.set_title(f"{window['name']} spectra", fontsize=9)
    ax.legend(fontsize=7)
plt.show()

### Against the simulation

In the continuum channels the line cube should hold noise only: the table compares the largest absolute value of the line cube in the continuum channels, inside the primary beam, with its noise (the standard deviation of its continuum channels in a corner away from the sources), and gives the continuum the subtraction removed at the two source positions. In the line channels the line cube should equal the dirty cube less its continuum: the last row is the largest difference between the line cube and the dirty cube less the continuum fitted to its own spectrum, at the phase centre, in units of the noise.

In [ ]:
def channel_mask(ranges, n_channels):
    mask = np.zeros(n_channels, dtype=bool)
    for start, stop in ranges:
        mask[start:stop] = True
    return mask


inner = slice(32, 96)
check = {}
for index, (window, window_range) in enumerate(
    zip(WINDOWS, window_ranges, strict=True)
):
    window_before = before.isel(frequency=slice(*window_range)).values
    window_after = after.isel(frequency=slice(*window_range)).values
    n_channels = window_range[1] - window_range[0]
    continuum = channel_mask(
        continuum_finding[index]["continuum_channel_ranges"], n_channels
    )
    noise = float(np.std(window_after[continuum, :20, :20]))
    centre = pixel_of(0.0, 0.0)
    source = pixel_of(*CONTINUUM[1]["offset_arcsec"])
    spectrum = window_before[:, centre[0], centre[1]]
    order = fits[window["name"]]["fit_order"]
    x = np.arange(n_channels)
    model = np.polyval(np.polyfit(x[continuum], spectrum[continuum], order), x)
    check[window["name"]] = {
        "noise of the line cube (mJy/beam)": round(1e3 * noise, 3),
        "line cube, continuum channels: max |I| / noise": round(
            float(np.abs(window_after[continuum][:, inner, inner]).max()) / noise, 2
        ),
        "continuum removed at the phase centre (mJy/beam)": round(
            1e3
            * float(
                np.mean(
                    spectrum[continuum] - window_after[continuum, centre[0], centre[1]]
                )
            ),
            2,
        ),
        "continuum removed at the source with spectral index (mJy/beam)": round(
            1e3
            * float(
                np.mean(
                    window_before[continuum, source[0], source[1]]
                    - window_after[continuum, source[0], source[1]]
                )
            ),
            2,
        ),
        "line channels: max |line cube - (dirty cube - fit)| / noise": round(
            float(
                np.abs(
                    window_after[~continuum, centre[0], centre[1]]
                    - (spectrum - model)[~continuum]
                ).max()
            )
            / noise,
            2,
        ),
    }
table(check)

### Comparison with CASA uvcontsub

CASA `uvcontsub` (`fitmethod="gsl"`, the pipeline's) fits the same channels of the first window, written to a Measurement Set v2: its continuum channels are the same at every time, so a CASA channel selection gives them exactly. The table gives the largest difference of the continuum-subtracted visibilities relative to the largest visibility, and the run times on one thread: CASA's task (reading and writing the Measurement Set) and AstroVIPER's processing function on the loaded visibilities. The data set is small, so the times are dominated by fixed costs. The cell is skipped without `casatasks`, `arcae` and `threadpoolctl`.

In [ ]:
import time

try:
    import arcae  # noqa: F401 -- the Measurement Set v2 writer
    from casatasks import uvcontsub
    from casatools import table as casa_table
    from threadpoolctl import threadpool_limits
except ImportError:
    uvcontsub = None
    print(
        "casatasks, arcae or threadpoolctl is not installed: the comparison with "
        "CASA is skipped."
    )

if uvcontsub is not None:
    from xradio.measurement_set import load_processing_set

    from astroviper.processing_functions.visibility_manipulation import (
        continuum_channel_mask,
    )
    from astroviper.processing_functions.visibility_manipulation import (
        uv_continuum_subtraction as uv_continuum_subtraction_function,
    )
    from astroviper.utils.measurement_set_v2 import write_measurement_set_v2

    name = WINDOWS[0]["name"]
    ms_name = f"ALMA_{name}"
    fit = fits[name]
    ms_v2 = "uv_continuum_subtraction_demo_spw_0.ms"
    write_measurement_set_v2(ps_store, ms_v2, ms_name=ms_name, overwrite=True)
    ms_xdt = load_processing_set(
        ps_store,
        sel_parms={ms_name: {}},
        data_group_name="base",
        include_variables=["VISIBILITY", "FLAG", "WEIGHT"],
    )[ms_name]
    mask = continuum_channel_mask(
        ms_xdt.frequency.values, fit["continuum_frequency_ranges"]
    )[0]
    edges = np.flatnonzero(np.diff(np.concatenate([[0], mask.astype(int), [0]])))
    selection = ";".join(
        f"{start}~{stop - 1}"
        for start, stop in zip(edges[::2], edges[1::2], strict=True)
    )
    casa_output = "uv_continuum_subtraction_demo_spw_0_casa.ms"
    shutil.rmtree(casa_output, ignore_errors=True)
    start = time.perf_counter()
    uvcontsub(
        vis=ms_v2,
        outputvis=casa_output,
        datacolumn="data",
        fitspec=f"0:{selection}",
        fitorder=fit["fit_order"],
    )
    casa_seconds = time.perf_counter() - start
    with threadpool_limits(limits=1):
        start = time.perf_counter()
        uv_continuum_subtraction_function(ms_xdt, overwrite=True, **fit)
        astroviper_seconds = time.perf_counter() - start

    tb = casa_table()
    tb.open(casa_output)
    casa_data = tb.getcol("DATA").T  # (row, channel, polarization)
    rows = np.stack(
        [tb.getcol("TIME"), tb.getcol("ANTENNA1"), tb.getcol("ANTENNA2")], axis=1
    )
    tb.close()
    tb.open(ms_v2)
    input_rows = np.stack(
        [tb.getcol("TIME"), tb.getcol("ANTENNA1"), tb.getcol("ANTENNA2")], axis=1
    )
    tb.close()
    assert np.array_equal(rows, input_rows), "CASA reordered the rows"
    ours = ms_xdt.VISIBILITY_CONTINUUM_SUBTRACTED.values.reshape(casa_data.shape)
    scale = np.abs(ms_xdt.VISIBILITY.values).max()
    display(
        pd.Series(
            {
                "continuum channels (CASA selection)": f"0:{selection}",
                "fit order": fit["fit_order"],
                "max |AstroVIPER - CASA| / max |V|": f"{np.abs(ours - casa_data).max() / scale:.2e}",
                "CASA uvcontsub, 1 thread (s)": round(casa_seconds, 3),
                "AstroVIPER processing function, 1 thread (s)": round(
                    astroviper_seconds, 3
                ),
                "visibility samples": ours.size,
            }
        ).to_frame("value")
    )

### Summary

`find_continuum_ultra` finds the continuum channels of each window in the dirty cube; `continuum_fit_from_find_continuum` turns them into the continuum fit of `hif_uvcontsub`; `uv_continuum_subtraction` subtracts the continuum of every visibility spectrum and writes the line data as a new data group. The line cube has noise in the continuum channels and the lines in their channels, and the continuum-subtracted visibilities agree with CASA `uvcontsub` to single precision.

In [ ]:
viper_client.close()